In [2]:
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# Настройки для красивых графиков
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


DATA_DIR = "./Data/"
MODELS_DIR = "./models/"

In [3]:
car_data = pd.read_csv(
    DATA_DIR + "car_data_preprocessed.csv", parse_dates=["Date"], index_col="Date"
)
sneakers_data = pd.read_csv(
    DATA_DIR + "sneakers_data_preprocessed.csv", parse_dates=["Date"], index_col="Date"
)

# Сплит + нормализация

In [4]:
SPLIT_RATIOS = (0.7, 0.15, 0.15)  # (train, val, test)

# Колонки, которые НЕ нужно нормализовать (бинарные флаги уже в [0, 1])
BINARY_COLS = [
    "is_month_start",
    "is_month_end",
    "is_quarter_start",
    "is_quarter_end",
    "is_year_start",
    "is_year_end",
    "is_friday",
    "is_monday",
]


# TODO прикрутить кроссвалидацию,а не просто статический сплит
def chronological_split(df, ratios):
    """
    Разбивает датафрейм хронологически на train/val/test.

    Параметры:
        df: датафрейм с DatetimeIndex
        ratios: кортеж (train_ratio, val_ratio, test_ratio)

    Возвращает:
        (train_df, val_df, test_df)
    """
    assert sum(ratios) == 1.0, f"Сумма ratios должна быть 1.0, получено {sum(ratios)}"

    n = len(df)
    train_end = int(n * ratios[0])
    val_end = train_end + int(n * ratios[1])

    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]

    return train, val, test


# TODO подумать, хороший ли вариант нормализовать именно минмаксом, возможно стоит взять какой-то другой scaler
def normalize_datasets(train, val, test, binary_cols):
    """
    Fit MinMaxScaler на train, transform на train/val/test.
    Бинарные колонки не нормализуются.

    Возвращает:
        (train_norm, val_norm, test_norm, scaler, cols_to_scale)
    """
    # Определяем колонки для нормализации (все числовые, кроме бинарных)
    all_numeric = train.select_dtypes(include=[np.number]).columns.tolist()
    cols_to_scale = [c for c in all_numeric if c not in binary_cols]
    cols_to_keep = [c for c in train.columns if c not in cols_to_scale]

    print(f"      Колонок для нормализации: {len(cols_to_scale)}")
    print(f"      Колонок без изменений: {len(cols_to_keep)}")

    # Fit на TRAIN только!
    scaler = MinMaxScaler()
    scaler.fit(train[cols_to_scale])

    # Transform всех трёх частей
    train_scaled = pd.DataFrame(
        scaler.transform(train[cols_to_scale]), columns=cols_to_scale, index=train.index
    )
    val_scaled = pd.DataFrame(
        scaler.transform(val[cols_to_scale]), columns=cols_to_scale, index=val.index
    )
    test_scaled = pd.DataFrame(
        scaler.transform(test[cols_to_scale]), columns=cols_to_scale, index=test.index
    )

    # Добавляем обратно ненормализованные колонки
    if cols_to_keep:
        train_scaled = pd.concat([train_scaled, train[cols_to_keep]], axis=1)
        val_scaled = pd.concat([val_scaled, val[cols_to_keep]], axis=1)
        test_scaled = pd.concat([test_scaled, test[cols_to_keep]], axis=1)

    # Восстанавливаем порядок колонок как в оригинале
    train_scaled = train_scaled[train.columns]
    val_scaled = val_scaled[train.columns]
    test_scaled = test_scaled[train.columns]

    return train_scaled, val_scaled, test_scaled, scaler, cols_to_scale


In [5]:
car_train, car_val, car_test = chronological_split(car_data, SPLIT_RATIOS)
car_train_norm, car_val_norm, car_test_norm, car_scaler, car_scaled_cols = (
    normalize_datasets(car_train, car_val, car_test, BINARY_COLS)
)

sneakers_train, sneakers_val, sneakers_test = chronological_split(
    sneakers_data, SPLIT_RATIOS
)
(
    sneakers_train_norm,
    sneakers_val_norm,
    sneakers_test_norm,
    sneakers_scaler,
    sneakers_scaled_cols,
) = normalize_datasets(sneakers_train, sneakers_val, sneakers_test, BINARY_COLS)

      Колонок для нормализации: 118
      Колонок без изменений: 8
      Колонок для нормализации: 62
      Колонок без изменений: 8


In [6]:
# Scaler'ы (для обратного преобразования при подсчёте метрик)
joblib.dump(car_scaler, MODELS_DIR + "car_scaler.pkl")
joblib.dump(sneakers_scaler, MODELS_DIR + "sneakers_scaler.pkl")

['./models/sneakers_scaler.pkl']

In [7]:
sneakers_data.columns

Index(['Nike', 'Puma', 'Nike_target', 'Puma_target', 'Nike_target_lag1',
       'Nike_target_lag2', 'Nike_target_lag3', 'Nike_target_lag4',
       'Nike_target_lag5', 'Nike_target_lag6', 'Nike_target_lag7',
       'Nike_target_lag8', 'Nike_target_lag9', 'Nike_target_lag10',
       'Nike_target_rolling_mean_5', 'Nike_target_rolling_std_5',
       'Nike_target_rolling_min_5', 'Nike_target_rolling_max_5',
       'Nike_target_rolling_mean_10', 'Nike_target_rolling_std_10',
       'Nike_target_rolling_min_10', 'Nike_target_rolling_max_10',
       'Nike_target_rolling_mean_20', 'Nike_target_rolling_std_20',
       'Nike_target_rolling_min_20', 'Nike_target_rolling_max_20',
       'Nike_target_rolling_mean_50', 'Nike_target_rolling_std_50',
       'Nike_target_rolling_min_50', 'Nike_target_rolling_max_50',
       'Puma_target_lag1', 'Puma_target_lag2', 'Puma_target_lag3',
       'Puma_target_lag4', 'Puma_target_lag5', 'Puma_target_lag6',
       'Puma_target_lag7', 'Puma_target_lag8', 'Puma_ta

In [8]:
car_data.columns

Index(['Volkswagen', 'Renault', 'Li Auto', 'NIO', 'Volkswagen_target',
       'Renault_target', 'Li Auto_target', 'NIO_target',
       'Volkswagen_target_lag1', 'Volkswagen_target_lag2',
       ...
       'day_of_year', 'week_of_year', 'is_month_start', 'is_month_end',
       'is_quarter_start', 'is_quarter_end', 'is_year_start', 'is_year_end',
       'is_friday', 'is_monday'],
      dtype='str', length=126)